# Steam single-player pricing analysis

**Business question:** What list price is associated with stronger estimated sales on Steam, and which modeling approach best predicts an ownership proxy from price and game metadata?

This notebook analyzes [`data/single-player-games.csv`](data/single-player-games.csv) (paid, single-player titles enriched with SteamSpy). It cleans the data, explores distributions, fits **multiple regression models**, and interprets results for a **non-technical stakeholder** in the findings sections.

| Section | Content |
|---------|---------|
| 1–7 | Configuration, cleaning, feature engineering (`genre__*`, `tag__*`) |
| 8 | EDA visualizations (Matplotlib + Seaborn) |
| 9 | Linear baseline + revenue-proxy “optimal price” scenario |
| 10 | Model comparison (7 algorithms, cross-validation / grid search) |
| 11 | Executive summary, recommendations, next steps |

**Target variable:** `log_owners_mid` — log of the midpoint of SteamSpy’s **estimated owner range** (a sales proxy, not verified unit sales).

**Primary evaluation metrics (regression):**
- **R² (coefficient of determination)** — share of variance in log owners explained on a held-out 20% test set; higher is better for ranking models.
- **RMSE / MAE** — typical prediction error on the log scale; lower is better.

**Important limitation:** Results describe **historical associations** in a cross-section of games—not causal proof that changing price will change sales.

**Outputs (local):** `data/single-player-games-cleaned.parquet` and plots in `visualizations/` (see [README](README.md)).

## 1. Configuration

Adjust paths and knobs here before running the rest.

In [1]:
from pathlib import Path

SCRIPT_DIR = Path.cwd()
INPUT_CSV = SCRIPT_DIR / "data" / "single-player-games.csv"
OUTPUT_PARQUET = SCRIPT_DIR / "data" / "single-player-games-cleaned.parquet"
OUTPUT_CSV = SCRIPT_DIR / "data" / "single-player-games-cleaned.csv"
WRITE_CSV_MIRROR = False  # set True for spreadsheet mirror (not committed; see .gitignore)

# None = today UTC midnight; or set e.g. "2026-05-05" for reproducible age_days
REFERENCE_DATE = None

GENRE_TOP_K = 20
TAG_TOP_K = 15
MIN_OWNERS_MID = None  # e.g. 5000.0 to drop very small estimates


## 2. Imports and helpers

The next cell installs `numpy`, `pandas`, and `pyarrow` into the **active kernel** if they are missing (fixes `ModuleNotFoundError` when the kernel is not your project `.venv`).

In [2]:
import importlib.util
import json
import re
import subprocess
import sys
from collections import Counter

_missing = ("numpy", "pandas", "pyarrow", "matplotlib", "seaborn")
if any(importlib.util.find_spec(p) is None for p in _missing):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

OWNERS_PATTERN = re.compile(r"([\d,]+)\s*\.\.\s*([\d,]+)")


def _strip_commas_num(s: str) -> int:
    return int(s.replace(",", "").strip())


def parse_owners_range(raw: object) -> tuple[float | None, float | None]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None, None
    text = str(raw).strip()
    if not text:
        return None, None
    m = OWNERS_PATTERN.search(text)
    if not m:
        return None, None
    try:
        low = _strip_commas_num(m.group(1))
        high = _strip_commas_num(m.group(2))
    except ValueError:
        return None, None
    if high < low:
        low, high = high, low
    return float(low), float(high)


def split_semicolon_counts(series: pd.Series) -> pd.Series:
    def count_parts(x: object) -> int:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return 0
        parts = [p.strip() for p in str(x).split(";") if p.strip()]
        return len(parts)

    return series.map(count_parts)


def split_genre_list(raw: object) -> list[str]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    return [g.strip() for g in str(raw).split(",") if g.strip()]


def sanitize_feature_name(s: str) -> str:
    out = re.sub(r"[^\w]+", "_", s.strip())
    out = re.sub(r"_+", "_", out).strip("_")
    return out or "unknown"


def parse_tags_dict(raw: object) -> dict:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return {}
    text = str(raw).strip()
    if not text:
        return {}
    try:
        data = json.loads(text)
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}


## 3. Load raw CSV

In [3]:
df = pd.read_csv(INPUT_CSV, dtype={"appid": "Int64"})
n_read = len(df)
print(f"Loaded {n_read} rows from {INPUT_CSV}")
df.head(2)


Loaded 10739 rows from /Users/mateovargas/Documents/Development/steam-price-analysis/data/single-player-games.csv


,appid,name,release_date,release_date_iso,price_currency,price_final_cents,price_initial_cents,price_discount_percent,price_final_formatted,developers,...,steamspy_average_forever,steamspy_average_2weeks,steamspy_median_forever,steamspy_median_2weeks,steamspy_score_rank,steamspy_price,steamspy_initialprice,steamspy_discount,steamspy_genre,steamspy_tags
0,1313,SiN Gold,"Mar 18, 2020",2020-03-18,USD,999,999,0,$9.99,Ritual Entertainment; Nightdive Studios,...,0.0,0.0,0.0,0.0,NaN,999.0,999.0,0.0,Action,"{""Action"": 167, ""FPS"": 29, ""Cult Classic"": 28,..."
1,7800,Stubbs the Zombie in Rebel Without a Pulse,"Mar 16, 2021",2021-03-16,USD,1999,1999,0,$19.99,Aspyr,...,0.0,0.0,0.0,0.0,NaN,1999.0,1999.0,0.0,Action,"{""Zombies"": 232, ""Funny"": 211, ""Villain Protag..."


## 4. Parse features (owners, dates, price, tags, genres)

In [4]:
# Naive datetimes only (ISO dates have no tz). Avoid tz-aware utcnow() vs naive release_dt.
ref = (
    pd.Timestamp(REFERENCE_DATE).normalize()
    if REFERENCE_DATE
    else pd.Timestamp(pd.Timestamp.now(tz="UTC").date())
)

low_list: list[float | None] = []
high_list: list[float | None] = []
for v in df["steamspy_owners"]:
    lo, hi = parse_owners_range(v)
    low_list.append(lo)
    high_list.append(hi)

df["owners_low"] = low_list
df["owners_high"] = high_list
df["owners_mid"] = (df["owners_low"] + df["owners_high"]) / 2.0
df["log_owners_mid"] = np.where(
    df["owners_mid"].notna() & (df["owners_mid"] > 0),
    np.log(df["owners_mid"]),
    np.nan,
)

df["release_dt"] = pd.to_datetime(df["release_date_iso"], errors="coerce")
df["release_year"] = df["release_dt"].dt.year
df["age_days"] = (ref - df["release_dt"]).dt.days

df["price_usd"] = df["price_final_cents"].astype("float64") / 100.0

df["developer_count"] = split_semicolon_counts(df["developers"])
df["publisher_count"] = split_semicolon_counts(df["publishers"])

tags_parsed = df["steamspy_tags"].map(parse_tags_dict)
df["tag_count"] = tags_parsed.map(len)
df["has_tags"] = df["tag_count"] > 0

df["genre_list"] = df["steamspy_genre"].map(split_genre_list)
df["primary_genre"] = df["genre_list"].map(lambda xs: xs[0] if xs else np.nan)


## 5. Filter rows (sequential drops)

In [5]:
mask_owners = df["owners_low"].notna() & df["owners_high"].notna()
n_bad_owners = int((~mask_owners).sum())
df = df.loc[mask_owners].copy()

mask_price = df["price_final_cents"].notna()
n_bad_price = int((~mask_price).sum())
df = df.loc[mask_price].copy()

mask_date = df["release_dt"].notna()
n_bad_date = int((~mask_date).sum())
df = df.loc[mask_date].copy()

if MIN_OWNERS_MID is not None:
    mask_min = df["owners_mid"] >= MIN_OWNERS_MID
    n_min_owners = int((~mask_min).sum())
    df = df.loc[mask_min].copy()
else:
    n_min_owners = 0

print(f"Reference date (age_days): {ref.date()}")
print(f"Dropped (unparseable owners): {n_bad_owners}")
print(f"Dropped (missing price): {n_bad_price}")
print(f"Dropped (invalid release date): {n_bad_date}")
if MIN_OWNERS_MID is not None:
    print(f"Dropped (owners_mid < {MIN_OWNERS_MID}): {n_min_owners}")
print(f"Remaining rows: {len(df)}")


Reference date (age_days): 2026-05-30
Dropped (unparseable owners): 3
Dropped (missing price): 0
Dropped (invalid release date): 0
Remaining rows: 10736


## 6. Top-K genre multi-hot columns

In [6]:
k = max(0, GENRE_TOP_K)
genre_counter: Counter[str] = Counter()
for lst in df["genre_list"]:
    for g in lst:
        genre_counter[g] += 1
top_genres = [g for g, _ in genre_counter.most_common(k)]

used_names: dict[str, str] = {}
genre_cols: list[str] = []
for g in top_genres:
    base = sanitize_feature_name(g)
    name = base
    i = 2
    while name in used_names.values():
        name = f"{base}_{i}"
        i += 1
    used_names[g] = name
    col = f"genre__{name}"
    genre_cols.append(col)
    df[col] = df["genre_list"].map(lambda lst, gg=g: int(gg in lst))

df = df.drop(columns=["genre_list"])
print(f"Genre multi-hot columns: {len(genre_cols)}")


Genre multi-hot columns: 18


## 6b. Top-K Steam tag multi-hot columns

In [7]:
k_tag = max(0, TAG_TOP_K)
tag_counter: Counter[str] = Counter()
for raw in df["steamspy_tags"]:
    for tag in parse_tags_dict(raw):
        tag_counter[tag] += 1
top_tags = [t for t, _ in tag_counter.most_common(k_tag)]

used_tag_names: dict[str, str] = {}
tag_cols: list[str] = []
for t in top_tags:
    base = sanitize_feature_name(t)
    name = base
    i = 2
    while name in used_tag_names.values():
        name = f"{base}_{i}"
        i += 1
    used_tag_names[t] = name
    col = f"tag__{name}"
    tag_cols.append(col)
    df[col] = df["steamspy_tags"].map(
        lambda raw, tt=t: int(tt in parse_tags_dict(raw))
    )

df_clean = df
print(f"Tag multi-hot columns: {len(tag_cols)}")

Tag multi-hot columns: 15


## 7. Validate, save, quick peek

**Column glossary (added / key fields):**

| Column | Meaning |
|--------|---------|
| `owners_mid`, `log_owners_mid` | Midpoint of SteamSpy owner range; log for skewed targets |
| `price_usd`, `price_discount_percent` | Store price and discount |
| `release_dt`, `release_year`, `age_days` | Parsed release date and age vs reference |
| `steamspy_ccu`, `steamspy_median_*` | Engagement (many zeros are normal) |
| `developer_count`, `publisher_count` | Count of `;`-separated names |
| `tag_count`, `has_tags` | Parsed JSON tag dict size |
| `primary_genre` | First genre in SteamSpy genre string |
| `genre__*` | Multi-hot for top-K genres |
| `tag__*` | Multi-hot for top-K SteamSpy tags |


In [8]:
required = [
    "appid",
    "owners_mid",
    "log_owners_mid",
    "price_usd",
    "price_discount_percent",
    "release_dt",
    "age_days",
]
missing = [c for c in required if c not in df_clean.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Wrote Parquet: {OUTPUT_PARQUET} ({len(df_clean)} rows)")

if WRITE_CSV_MIRROR:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote CSV: {OUTPUT_CSV}")

df_clean[["name", "owners_mid", "price_usd", "age_days", "primary_genre"]].head()


Wrote Parquet: /Users/mateovargas/Documents/Development/steam-price-analysis/data/single-player-games-cleaned.parquet (10736 rows)


,name,owners_mid,price_usd,age_days,primary_genre
0,SiN Gold,150000.0,9.99,2264,Action
1,Stubbs the Zombie in Rebel Without a Pulse,350000.0,19.99,1901,Action
2,Second Sight,35000.0,9.99,1878,Action
3,Grand Theft Auto IV: The Complete Edition,7500000.0,19.99,2258,Action
4,Big Mutha Truckers,10000.0,8.99,356,Racing


In [9]:
# Summary stats for key numeric fields (print avoids large HTML table output)
print(
    df_clean[["owners_mid", "price_usd", "age_days", "steamspy_ccu"]]
    .describe()
    .round(2)
    .to_string()
)


        owners_mid  price_usd  age_days  steamspy_ccu
count     10736.00   10736.00  10736.00      10736.00
mean     142911.70       9.30   3057.19         47.69
std      803409.34       9.75    590.04        718.57
min       10000.00       0.49     30.00          0.00
25%       10000.00       2.99   2987.00          0.00
50%       10000.00       5.99   3186.00          0.00
75%       35000.00      12.99   3411.25          0.00
max    35000000.00     199.99   3676.00      32112.00


## 8. Visualizations (saved to `visualizations/`)

Exploratory plots before modeling (Matplotlib + Seaborn):

- **Continuous variables:** price and log-owner histograms; scatter plots (price vs owners, discount vs owners).
- **Categorical variable:** boxplot of price by `primary_genre`; bar chart of mean log owners by genre.
- **Multivariate:** correlation heatmap of numeric features.

All plots use descriptive titles and human-readable axis labels. Figures are saved to `visualizations/` (regenerate by running this section).


In [10]:
from pathlib import Path

VIS_DIR = SCRIPT_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("Saving plots to visualizations/")

# If you open this notebook without re-running earlier cells, reload the cleaned dataset.
if "df_clean" not in globals():
    df_clean = pd.read_parquet(OUTPUT_PARQUET)

# Convenience columns
_df = df_clean.copy()
_df["log_price_usd"] = np.where(_df["price_usd"] > 0, np.log(_df["price_usd"]), np.nan)

# Robust y for plotting (avoid inf)
_df["log_owners_mid"] = np.where(_df["owners_mid"] > 0, np.log(_df["owners_mid"]), np.nan)

# Limit extreme owners for clearer scatter (keep full data for modeling)
owners_cap = _df["owners_mid"].quantile(0.995)
_df_scatter = _df[_df["owners_mid"] <= owners_cap].copy()
print(f"Scatter cap owners_mid at p99.5={owners_cap:,.0f}; rows kept={len(_df_scatter)}")


Saving plots to visualizations/
Scatter cap owners_mid at p99.5=3,500,000; rows kept=10702


In [11]:
# 1) Price distribution
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(_df["price_usd"], bins=60, kde=False, ax=ax)
ax.set_title("Distribution of Steam list prices (USD)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Number of games")
fig.tight_layout()
fig.savefig(VIS_DIR / "price_usd_hist.png", dpi=200)
plt.close(fig)

# 2) Owners distribution (log scale)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(_df["log_owners_mid"].dropna(), bins=60, kde=False, ax=ax)
ax.set_title("Distribution of log estimated owners (SteamSpy midpoint)")
ax.set_xlabel("Log estimated owners (midpoint)")
ax.set_ylabel("Number of games")
fig.tight_layout()
fig.savefig(VIS_DIR / "owners_mid_log_hist.png", dpi=200)
plt.close(fig)

# 3) Price vs owners (hexbin-like view using seaborn scatter with alpha)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    _df_scatter["price_usd"],
    _df_scatter["log_owners_mid"],
    s=10,
    alpha=0.15,
    linewidths=0,
)
ax.set_title("List price vs estimated owners (outliers capped at 99.5th percentile)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Log estimated owners (midpoint)")
fig.tight_layout()
fig.savefig(VIS_DIR / "price_vs_log_owners_scatter.png", dpi=200)
plt.close(fig)

# 4) Discount vs owners
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    _df_scatter["price_discount_percent"],
    _df_scatter["log_owners_mid"],
    s=10,
    alpha=0.15,
    linewidths=0,
)
ax.set_title("Discount percent vs estimated owners (outliers capped)")
ax.set_xlabel("Discount (%)")
ax.set_ylabel("Log estimated owners (midpoint)")
fig.tight_layout()
fig.savefig(VIS_DIR / "discount_vs_log_owners_scatter.png", dpi=200)
plt.close(fig)

print("Saved: price_usd_hist.png, owners_mid_log_hist.png, price_vs_log_owners_scatter.png, discount_vs_log_owners_scatter.png")


Saved: price_usd_hist.png, owners_mid_log_hist.png, price_vs_log_owners_scatter.png, discount_vs_log_owners_scatter.png


In [12]:
# 5) Boxplot: price by top genres (primary_genre)
_top_genres = (
    _df["primary_genre"].value_counts(dropna=True).head(12).index.tolist()
)
_df_gen = _df[_df["primary_genre"].isin(_top_genres)].copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=_df_gen,
    x="primary_genre",
    y="price_usd",
    ax=ax,
    showfliers=False,
)
ax.set_title("List price by primary genre (top 12 genres)")
ax.set_xlabel("Primary genre")
ax.set_ylabel("Price (USD)")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(VIS_DIR / "price_by_primary_genre_box.png", dpi=200)
plt.close(fig)

# 6) Mean log owners by primary genre (top 12)
fig, ax = plt.subplots(figsize=(10, 5))
mean_by_genre = (
    _df_gen.groupby("primary_genre")["log_owners_mid"].mean().sort_values(ascending=False)
)
mean_by_genre.plot(kind="bar", ax=ax)
ax.set_title("Average estimated owners by primary genre (top 12)")
ax.set_xlabel("Primary genre")
ax.set_ylabel("Mean log estimated owners")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(VIS_DIR / "mean_log_owners_by_primary_genre_bar.png", dpi=200)
plt.close(fig)

print("Saved: price_by_primary_genre_box.png, mean_log_owners_by_primary_genre_bar.png")


Saved: price_by_primary_genre_box.png, mean_log_owners_by_primary_genre_bar.png


In [13]:
# 7) Correlation heatmap for numeric features
num_cols = [
    "owners_mid",
    "log_owners_mid",
    "price_usd",
    "price_discount_percent",
    "age_days",
    "steamspy_ccu",
    "steamspy_average_forever",
    "steamspy_average_2weeks",
    "steamspy_median_forever",
    "steamspy_median_2weeks",
    "developer_count",
    "publisher_count",
    "tag_count",
]
num_cols = [c for c in num_cols if c in _df.columns]

corr = _df[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    cmap="vlag",
    center=0,
    annot=False,
    square=False,
    ax=ax,
)
ax.set_title("Correlation heatmap of numeric features")
fig.tight_layout()
fig.savefig(VIS_DIR / "correlation_heatmap_numeric.png", dpi=200)
plt.close(fig)

print("Saved: correlation_heatmap_numeric.png")


Saved: correlation_heatmap_numeric.png


## 9. Linear regression: optimal price (reference game)

**Model:** ordinary least squares (OLS) predicting `log_owners_mid` from `price_usd` plus controls (`age_days`, tag/developer/publisher counts, top-K `genre__*` flags).

**Evaluation:** holdout **test R²**, **RMSE**, and **5-fold cross-validation R²** on the training split. R² is appropriate because we care how much of the variance in the ownership proxy is explained; RMSE/MAE quantify typical prediction error on the log scale.

**Business use:** sweep a **revenue proxy** (`price × predicted owners`) over a price grid for a synthetic reference game (median counts, mode genre)—exploratory only, not causal launch pricing.


In [14]:
import importlib.util
import subprocess
import sys
import warnings
from pathlib import Path

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

if "df_clean" not in globals():
    df_clean = pd.read_parquet(OUTPUT_PARQUET)

if "VIS_DIR" not in globals():
    VIS_DIR = SCRIPT_DIR / "visualizations"
    VIS_DIR.mkdir(parents=True, exist_ok=True)

genre_cols = [c for c in df_clean.columns if c.startswith("genre__")]
feature_cols = [
    "price_usd",
    "age_days",
    "tag_count",
    "developer_count",
    "publisher_count",
    *genre_cols,
]
model_df = df_clean.dropna(
    subset=["log_owners_mid", "price_usd", *feature_cols[1:]]
).copy()

X = model_df[feature_cols].astype(np.float64)
y = model_df["log_owners_mid"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


def predict_log_owners(model, X_df: pd.DataFrame) -> np.ndarray:
    """Predict without BLAS RuntimeWarning noise from ill-conditioned matmul."""
    X_arr = X_df[feature_cols].astype(np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
            return model.predict(X_arr)


reg = LinearRegression(fit_intercept=True)
reg.fit(X_train, y_train)

y_train_pred = predict_log_owners(reg, X_train)
y_test_pred = predict_log_owners(reg, X_test)
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

rmse_test = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
mae_test = mean_absolute_error(y_test, y_test_pred)
cv_r2 = cross_val_score(reg, X_train, y_train, cv=5, scoring="r2").mean()

price_coef_idx = feature_cols.index("price_usd")
price_coef = reg.coef_[price_coef_idx]

print(f"Rows used: {len(model_df):,}")
print(f"R² (train): {r2_train:.4f}")
print(f"R² (test):  {r2_test:.4f}")
print(f"RMSE (test): {rmse_test:.4f}")
print(f"MAE (test):  {mae_test:.4f}")
print(f"5-fold CV R² (train): {cv_r2:.4f}")
print(f"price_usd coefficient: {price_coef:.6f}")

# Reference game: median counts, genre flags from mode primary_genre
ref = {}
for col in ["age_days", "tag_count", "developer_count", "publisher_count"]:
    ref[col] = float(X_train[col].median())

mode_genre = model_df["primary_genre"].mode(dropna=True).iloc[0]
mode_genre_col = f"genre__{sanitize_feature_name(str(mode_genre))}"
for col in genre_cols:
    ref[col] = 1.0 if col == mode_genre_col else 0.0

ref_row = {**ref, "price_usd": float(X_train["price_usd"].median())}
ref_X = pd.DataFrame([ref_row], columns=feature_cols)

p_lo = float(model_df["price_usd"].quantile(0.01))
p_hi = float(model_df["price_usd"].quantile(0.99))
price_grid = np.linspace(p_lo, p_hi, 200)

ref_base = ref_X.drop(columns=["price_usd"]).iloc[0]
grid_rows = []
for p in price_grid:
    row = ref_base.copy()
    row["price_usd"] = p
    grid_rows.append(row)
grid_X = pd.DataFrame(grid_rows, columns=feature_cols)

pred_log_owners = predict_log_owners(reg, grid_X)
pred_owners = np.exp(pred_log_owners)
revenue = price_grid * pred_owners

p_opt_idx = int(np.argmax(revenue))
p_opt = float(price_grid[p_opt_idx])
rev_opt = float(revenue[p_opt_idx])
owners_opt = float(pred_owners[p_opt_idx])

p_median = float(X_train["price_usd"].median())
ref_median = ref_X.copy()
ref_median["price_usd"] = p_median
log_at_median = float(predict_log_owners(reg, ref_median)[0])
owners_at_median = float(np.exp(log_at_median))
rev_at_median = p_median * owners_at_median

print()
print("Reference profile:")
print(f"  primary_genre (mode): {mode_genre}")
print(f"  age_days (median): {ref['age_days']:.0f}")
print(f"  tag_count (median): {ref['tag_count']:.0f}")
print()
print(f"Price grid: ${p_lo:.2f} – ${p_hi:.2f} (1st–99th percentile of catalog)")
print(f"Optimal price (revenue proxy max): ${p_opt:.2f}")
print(f"  Predicted owners_mid: {owners_opt:,.0f}")
print(f"  Predicted revenue proxy: ${rev_opt:,.0f}")
print()
print(f"At median catalog price (${p_median:.2f}) on same profile:")
print(f"  Predicted owners_mid: {owners_at_median:,.0f}")
print(f"  Predicted revenue proxy: ${rev_at_median:,.0f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(price_grid, revenue, color="steelblue", lw=2)
ax.axvline(p_opt, color="crimson", ls="--", lw=1.5, label=f"optimal ${p_opt:.2f}")
ax.axvline(
    p_median,
    color="gray",
    ls=":",
    lw=1.5,
    label=f"median catalog ${p_median:.2f}",
)
ax.set_title("Predicted revenue vs list price (typical reference game)")
ax.set_xlabel("List price (USD)")
ax.set_ylabel("Predicted revenue proxy (USD)")
ax.legend()
fig.tight_layout()
out_path = VIS_DIR / "optimal_price_revenue_curve.png"
fig.savefig(out_path, dpi=200)
plt.close(fig)
print("\nSaved to Visualisations.")

Rows used: 10,736
R² (train): 0.2888
R² (test):  0.2967
RMSE (test): 1.1852
MAE (test):  0.8885
5-fold CV R² (train): 0.2833
price_usd coefficient: 0.022967

Reference profile:
  primary_genre (mode): Action
  age_days (median): 3186
  tag_count (median): 8

Price grid: $0.89 – $39.99 (1st–99th percentile of catalog)
Optimal price (revenue proxy max): $39.99
  Predicted owners_mid: 58,806
  Predicted revenue proxy: $2,351,656

At median catalog price ($5.99) on same profile:
  Predicted owners_mid: 26,933
  Predicted revenue proxy: $161,330

Saved to Visualisations.


/Users/mateovargas/Documents/Development/steam-price-analysis/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/mateovargas/Documents/Development/steam-price-analysis/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/mateovargas/Documents/Development/steam-price-analysis/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/mateovargas/Documents/Development/steam-price-analysis/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/mateovargas/Documents/Development/steam-price-analysis/.venv/lib/python3.10/site-packages/sklearn/linear_model/_

### Findings (reference-game linear model)

**For stakeholders:** This model asks, *“At what list price does our formula predict the highest revenue for a typical Action game?”* The answer is sensitive to how price and owners move together in past data—it is **not** a recommendation to raise prices.

**Metrics:** Test **R² ≈ 0.30** means price and basic metadata explain only part of who sells well. **Positive price coefficient** usually reflects hit games being both expensive and popular (confounding), not that higher price causes more sales.

**Actionable insight:** Use the median catalog price (~**$6**) as a realistic anchor; treat any “optimum” at the top of the price grid (~**$40**) as a **model artifact**, not a business target.

**Chart:** `optimal_price_revenue_curve.png` — predicted revenue proxy vs price for the reference profile.


## 10. Model comparison

Compare **seven regression models** on the **full feature set** (`price_usd`, counts, `genre__*`, `tag__*`).

**Evaluation metrics (same 80/20 holdout, `random_state=42`):**
- **R²** — primary metric for ranking models (explained variance on log owners).
- **RMSE / MAE** — error on the log scale (lower is better).
- **5-fold CV R²** — stability check on the training split.

**Cross-validation & grid search used:**
- **Ridge / Lasso:** `RidgeCV` / `LassoCV` (5-fold CV over regularization strength).
- **KNN:** `GridSearchCV` over `n_neighbors` ∈ {5, 15, 31, 51} (3-fold CV).
- **Other models:** fixed sensible defaults; overall ranking uses holdout test R².


In [15]:
import importlib.util
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

for _warn_mod in (
    r"sklearn\.utils\.extmath",
    r"sklearn\.linear_model\._base",
    r"sklearn\.linear_model\._ridge",
):
    warnings.filterwarnings("ignore", category=RuntimeWarning, module=_warn_mod)


@contextmanager
def _quiet_sklearn_matmul():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
            yield


if "df_clean" not in globals():
    df_clean = pd.read_parquet(OUTPUT_PARQUET)

if "VIS_DIR" not in globals():
    VIS_DIR = SCRIPT_DIR / "visualizations"
    VIS_DIR.mkdir(parents=True, exist_ok=True)

genre_cols = [c for c in df_clean.columns if c.startswith("genre__")]
tag_cols = [c for c in df_clean.columns if c.startswith("tag__")]
if not tag_cols:
    raise ValueError(
        "No tag__ columns in df_clean. Re-run §6b and §7 to rebuild parquet with TAG_TOP_K."
    )

feature_cols = [
    "price_usd",
    "age_days",
    "tag_count",
    "developer_count",
    "publisher_count",
    *genre_cols,
    *tag_cols,
]

model_df = df_clean.dropna(
    subset=["log_owners_mid", "price_usd", *feature_cols[1:]]
).copy()
X = model_df[feature_cols].astype(np.float64)
y = model_df["log_owners_mid"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Rows: {len(model_df):,}  |  Features: {len(feature_cols)}")

MODELS: dict[str, object] = {
    "OLS": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", LinearRegression()),
        ]
    ),
    "Ridge": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                RidgeCV(alphas=np.logspace(-2, 4, 30), cv=5),
            ),
        ]
    ),
    "Lasso": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", LassoCV(cv=5, max_iter=5000, random_state=42)),
        ]
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "KNN": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                GridSearchCV(
                    KNeighborsRegressor(),
                    param_grid={"n_neighbors": [5, 15, 31, 51]},
                    cv=3,
                ),
            ),
        ]
    ),
    "MLP": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                MLPRegressor(
                    hidden_layer_sizes=(64, 32),
                    max_iter=500,
                    early_stopping=True,
                    random_state=42,
                ),
            ),
        ]
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
    ),
}


def _predict(model, X_df: pd.DataFrame) -> np.ndarray:
    X_in = X_df[feature_cols].astype(np.float64)
    with _quiet_sklearn_matmul():
        return model.predict(X_in)


# Train each model; record holdout metrics, CV R², and wall-clock time
results: list[dict] = []
for name, estimator in MODELS.items():
    t0 = time.perf_counter()
    with _quiet_sklearn_matmul():
        estimator.fit(X_train, y_train)
    seconds = time.perf_counter() - t0

    y_tr_pred = _predict(estimator, X_train)
    y_te_pred = _predict(estimator, X_test)
    cv_r2 = cross_val_score(estimator, X_train, y_train, cv=5, scoring="r2").mean()
    results.append(
        {
            "model": name,
            "r2_train": r2_score(y_train, y_tr_pred),
            "r2_test": r2_score(y_test, y_te_pred),
            "rmse_test": float(np.sqrt(mean_squared_error(y_test, y_te_pred))),
            "mae_test": mean_absolute_error(y_test, y_te_pred),
            "cv_r2": cv_r2,
            "seconds": seconds,
        }
    )
    print(f"  {name:<18} fit {seconds:6.1f}s  test R² {results[-1]['r2_test']:.4f}")

leaderboard = pd.DataFrame(results).sort_values("r2_test", ascending=False)
leaderboard = leaderboard.reset_index(drop=True)

print()
print("Model comparison (sorted by test R²):")
display_cols = ["model", "r2_test", "cv_r2", "rmse_test", "mae_test", "seconds"]
print(leaderboard[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

best = leaderboard.iloc[0]
print()
print(
    f"Best on holdout: {best['model']} "
    f"(test R²={best['r2_test']:.4f}, RMSE={best['rmse_test']:.4f}, "
    f"fit {best['seconds']:.1f}s)"
)

fig, ax = plt.subplots(figsize=(9, 5))
order = leaderboard.sort_values("r2_test", ascending=True)
ax.barh(order["model"], order["r2_test"], color="steelblue")
ax.set_xlabel("Test R² (held-out 20%)")
ax.set_title("Regression model comparison: predicted log(estimated owners)")
ax.set_xlim(0, max(0.05, order["r2_test"].max() * 1.15))
fig.tight_layout()
fig.savefig(VIS_DIR / "model_comparison_test_r2.png", dpi=200)
plt.close(fig)
print("\nSaved to Visualisations.")

Rows: 10,736  |  Features: 38
  OLS                fit    0.0s  test R² 0.3310
  Ridge              fit    0.2s  test R² 0.3298
  Lasso              fit    0.0s  test R² 0.3291
  RandomForest       fit    0.4s  test R² 0.4337
  KNN                fit    0.4s  test R² 0.3320
  MLP                fit    0.5s  test R² 0.3513
  GradientBoosting   fit    0.9s  test R² 0.4390

Model comparison (sorted by test R²):
           model  r2_test  cv_r2  rmse_test  mae_test  seconds
GradientBoosting   0.4390 0.4159     1.0585    0.7834   0.9383
    RandomForest   0.4337 0.4061     1.0635    0.7802   0.3833
             MLP   0.3513 0.3303     1.1382    0.8492   0.5303
             KNN   0.3320 0.3215     1.1551    0.8287   0.3559
             OLS   0.3310 0.3078     1.1559    0.8666   0.0045
           Ridge   0.3298 0.3082     1.1570    0.8676   0.1618
           Lasso   0.3291 0.3081     1.1576    0.8670   0.0245

Best on holdout: GradientBoosting (test R²=0.4390, RMSE=1.0585, fit 0.9s)

Saved to

### Findings (model comparison)

Re-run §10 to refresh numbers. On this dataset, **Gradient Boosting** and **Random Forest** typically achieve the highest **test R² (~0.43–0.44)**, beating linear models (~0.33). That suggests **nonlinear interactions** among price, genre, and tags matter for predicting the ownership proxy.

**For stakeholders:** Better prediction does **not** mean we know the perfect price. Tree models are harder to interpret for pricing decisions; keep **§9 OLS** for transparent “what-if” price curves.

**Actionable insight:** When forecasting interest for similar games, prefer the **top-ranked tree model** on R²/RMSE; when explaining pricing trade-offs to non-technical partners, use the **linear baseline** and clearly state assumptions.


## 11. Executive summary and recommendations

### Business problem
Indie and mid-tier publishers need evidence-informed **list prices** on Steam. We analyzed ~**10,700** paid single-player games to relate **list price and metadata** to SteamSpy’s **estimated owner counts** (a noisy stand-in for sales).

### Key findings
1. **Typical list price** is about **$6** (median); most games cluster below **$13**.
2. **Price alone** is only **modestly** correlated with estimated owners; genre, tags, and unobserved quality matter more.
3. **Linear “optimal price”** on catalog-wide data often lands at the **top of the observed price range**—a sign of confounding, not a pricing rule.
4. **Best predictive models** (Gradient Boosting / Random Forest, test R² ~**0.44**) beat simple linear models (~**0.33**) when genre and tag features are included.

### Recommendations (non-technical)
- **Do not** treat model output as “charge $X and succeed.” Use it to **frame discussions** and compare scenarios.
- **Do** benchmark against **similar games** (same genre, price band, tags) and real market comps.
- **Do** validate with **wishlists, demos, and soft launches** before committing to a final price.

### Next steps
- Collect **game-specific** comparables and refit models on that subset.
- Add **review scores** or **Metacritic** if available (not in current extract).
- Track **price changes over time** (sales events) for causal elasticity—not possible in this single snapshot.
- See [README.md](README.md) for project setup and a summary of results.
